In [1]:
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

**Dataset**

In [2]:
texts = [
    "I love this movie",
    "This film was amazing",
    "The acting was great",
    "I really enjoyed the story",
    "What a fantastic experience",
    "This was boring",
    "I hate this movie",
    "The plot was terrible",
    "Very bad acting",
    "I did not enjoy this film",
    "The movie was wonderful",
    "Absolutely loved it",
    "It was a great performance",
    "This was a horrible film",
    "Worst movie ever",
    "I liked the soundtrack",
    "The ending was beautiful",
    "The characters were boring",
    "I will watch it again",
    "It was a waste of time"
]

labels = [
    1, 1, 1, 1, 1,
    0, 0, 0, 0, 0,
    1, 1, 1, 0, 0,
    1, 1, 0, 1, 0
]

print("Total samples:", len(texts))
print("First sample:", texts[0], labels[0])

Total samples: 20
First sample: I love this movie 1


In [3]:
train_texts, val_texts, train_labels, val_labels = train_test_split(
    texts, labels, test_size=0.20, random_state=42
)

print("Train size:", len(train_texts))
print("Validation size:", len(val_texts))

Train size: 16
Validation size: 4


**Load tokenizer**

In [4]:
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

print(tokenizer)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

BertTokenizer(name_or_path='distilbert-base-uncased', vocab_size=30522, model_max_length=512, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	100: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	101: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	102: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	103: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}
)


In [5]:
sample_text = train_texts[0]

encoded_sample = tokenizer(
    sample_text,
    padding="max_length",
    truncation=True,
    max_length=16,
    return_tensors="pt"
)

print("Text:", sample_text)
print("Input IDs:", encoded_sample["input_ids"])
print("Attention Mask:", encoded_sample["attention_mask"])
print("Decoded back:", tokenizer.decode(encoded_sample["input_ids"][0]))

Text: Very bad acting
Input IDs: tensor([[ 101, 2200, 2919, 3772,  102,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0]])
Attention Mask: tensor([[1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]])
Decoded back: [CLS] very bad acting [SEP] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD]


**Custom Dataset class**

In [6]:
class SentimentDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=32):
        self.texts = texts
        self.lables = labels
        self.tokenizer = tokenizer
        self.max_length = max_length
    
    def len(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        text = self.texts[idx]
        label = self.lables[idx]
        
        encoding = tokenizer(
            text,
            padding="max_length",
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt"
        )
        
        item = {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "labels": torch.tensor(label, dtype=torch.long)
        }
        return item